In [ ]:
# Install required packages
!pip install --upgrade --quiet 'natural-pdf[ai,export]>=0.5.0'

print('✓ Packages installed!')

**Slides:** [slides.pdf](./slides.pdf)

# Let's ask questions

Time for some AI magic. Instead of just demanding accuracy we think like journalists: the goal is to **make verification as simple as possible.**

In [ ]:
from natural_pdf import PDF

pdf = PDF("https://github.com/jsoma/natural-pdf/raw/refs/heads/main/pdfs/01-practice.pdf")
page = pdf.pages[0]
page.show(width=900)

## Structured data generation

Usually if it's just a specific piece of text you're looking for, you can use spatial commands to pull it out. The times that LLMs come in handy is when there's a bit of nuance in your question (or the answer). You want it to write things that aren't in there, or piece together something complicated. It's worth the potential for hallucinations!

Below we're using Google thanks to its [OpenAI compatibility](https://ai.google.dev/gemini-api/docs/openai).

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY") or input("Enter GOOGLE_API_KEY: ")

In [ ]:
from openai import OpenAI

# Initialize your LLM client
# Anything OpenAI-compatible works!
client = OpenAI(
    api_key=GOOGLE_API_KEY,
    # api_key="YOUR_API_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"  # Changes based on what AI you're using
)

fields = ["site", "date", "violation count", "inspection service", "summary", "city", "full name of state"]
results = page.extract(fields, client=client, model="gemini-3.1-flash-lite")
results.to_dict()

### Confidence scores and citations

We get a few bonus treats, too: confidence scores and citations.

Interestingly enough, **confidence scores can sometimes decrease accuracy.** The LLMs make them up, *and* because they make the prompt so much more complicated accuracy always drops when you include it. I recommend not using it unless you're paying for a more expensive model.

**Citations are great**, though, especially when you'd like to be *accountable* and *responsible*.

In [ ]:
from openai import OpenAI

# Initialize your LLM client
# Anything OpenAI-compatible works!
client = OpenAI(
    api_key=GOOGLE_API_KEY,
    # api_key="YOUR_API_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"  # Changes based on what AI you're using
)

fields = ["site", "date", "inspection number", "violation count", "inspection service", "summary", "city", "full name of state"]
results = page.extract(fields,
                       instructions="You are parsing a document",
                       client=client,
                       model="gemini-3.1-flash-lite",
                       confidence=True,
                       citations=True)
results

In [ ]:
results.to_dict()

In [ ]:
# remove the confidences with confidences=False
results.to_dict(confidence=False)

Easily citations with `.show()`

In [ ]:
results.show()

### Very intense structured data extraction

Instead of being kind of loose and free with what you want, you can also get MUCH fancier and write a Pydantic model. It will not only send the column names you want, but also little descriptions and demands about strings (text), integers, floats and more.

You can find more details [here](https://platform.openai.com/docs/guides/structured-outputs).

In [ ]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI

# Initialize your LLM client
# Anything OpenAI-compatible works!
client = OpenAI(
    api_key=GOOGLE_API_KEY,
    # api_key="YOUR_API_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# Define your schema
class ReportInfo(BaseModel):
    inspection_number: str = Field(description="The main report identifier")
    inspection_date: str = Field(description="The name of the issuing company")
    inspection_service: str = Field(description="Name of inspection service")
    site: str = Field(description="Name of company inspected")
    summary: str = Field(description="Visit summary")
    city: str
    state: str = Field(description="Full name of state")
    violation_count: int

# Extract data
result = page.extract(schema=ReportInfo, client=client, model="gemini-3.1-flash-lite") 

In [ ]:
result

There are a handful of ways to access the results.

In [ ]:
result.to_dict()['inspection_date']

In [ ]:
result['inspection_date'].value

In [ ]:
result.data.inspection_date

## Table extraction with LLMs

In the example below, we're saying "Using Gemini, provide a violations table - each row should have a statute, a description, a level, and a repeat-checked

In [ ]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import List, Literal

client = OpenAI(
    api_key=GOOGLE_API_KEY,
    # api_key="YOUR_API_KEY_HERE",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

class ViolationsRow(BaseModel):
    statute: str
    description: str
    level: str
    repeat_checked: Literal["checked", "unchecked"] = Field("Whether the checkbox is checked or not")

class ViolationsTable(BaseModel):
    inspection_id: str
    violations: List[ViolationsRow]

result = page.extract(schema=ViolationsTable, client=client, model="gemini-3.1-flash-lite") 
result

Note that when we look below... **it didn't do the checked/unchecked correctly!**

In [ ]:
import pandas as pd

violations = result.to_dict()['violations']
pd.DataFrame(violations)

**This is why you can't trust LLMs**. When possible, extracting specifically from the page – especially with numbers! – is always your best approach.

# Putting things in categories

Sometimes you have documents that might fall into various categories - is it a police report? An interview transcript? A photograph? Whether you need to categorize based on how something *looks* or the text inside of it, Natural PDF has you covered!

## Categorizing an entire PDF

In [ ]:
from natural_pdf import PDF

pdf = PDF("https://github.com/jsoma/natural-pdf/raw/refs/heads/main/pdfs/01-practice.pdf")
page = pdf.pages[0]
page.show(width=500)

What can we classify the entire PDF as? Maybe a... slaughterhouse report? A dolphin training manual? Something about basketball or birding?

In [ ]:
pdf.category_confidence

## Classifying pages of a PDF

Let's take a look at a document from the CIA investigating whether you can **use pigeons as spies**.

In [ ]:
from natural_pdf import PDF

pdf = PDF("https://github.com/jsoma/ire25-natural-pdf/raw/refs/heads/main/cia-doc.pdf")
pdf.pages.show(cols=6)

Just like we did above, we can ask what category we think the PDF belongs to.

In [ ]:
pdf.classify(['slaughterhouse report', 'dolphin training manual', 'basketball', 'birding'], using='text')
(pdf.category, pdf.category_confidence)

But notice how all of the pages look very very different: **we can also categorize each page using vision**.

In [ ]:
pdf.classify_pages(['diagram', 'text', 'invoice', 'blank'], using='vision')

for page in pdf.pages:
    print(f"Page {page.number} is {page.category} - {page.category_confidence:0.3}")

And if we just want to see the pages that are diagrams, we can `.filter` for them.

In [ ]:
(
    pdf.pages
    .filter(lambda page: page.category == 'diagram')
    .show(show_category=True)
)


And if that's all we're interested in? We can save a new PDF of just those pages!

In [ ]:
(
    pdf.pages
    .filter(lambda page: page.category == 'diagram')
    .save_pdf("diagrams.pdf", original=True)
)